# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}\n{metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# Get record set information from metadata
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}")

for record_set in record_sets:
    print("RecordSet @id:", record_set.id)
    print("  Name:", record_set.name)
    print("  Description:", record_set.description)
    # List all fields and their @ids
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.id} (Type: {field.data_type})")
    print()

## 3. Data Extraction
Load data from each record set into DataFrames for further analysis. All entities are referenced by their `@id`.

In [ ]:
# Gather record set @id values
record_set_ids = [rset.id for rset in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f'Extracting data from record set: {record_set_id}')
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'Fields (columns): {list(df.columns)}')
        display(df.head())
    else:
        print(f'No records found for {record_set_id}')

## 4. Exploratory Data Analysis (EDA)
Apply basic processing steps: filter, normalize, and aggregate data on selected fields. All field names are referenced by their `@id`.

In [ ]:
# Select a record set and a numeric field by their @id
# For demonstration, choose the first available record set & a numeric field:
example_record_set_id = None
numeric_field = None
group_field = None

# Try to find a record set with at least one numeric field and one potential categorical field
import numpy as np
for rset in dataset.metadata.record_sets:
    df = dataframes.get(rset.id, None)
    if df is not None and not df.empty:
        for field in rset.fields:
            # Heuristic: Find a numeric field
            if "Float" in field.data_type or "Integer" in field.data_type or field.data_type.lower() in ['number', 'float', 'integer']:
                if field.id in df.columns:
                    numeric_field = field.id
                    example_record_set_id = rset.id
                    break
        if numeric_field:
            # Try to find a grouping field that's not numeric
            for field in rset.fields:
                if field.id != numeric_field and ("Text" in field.data_type or field.data_type.lower() in ['text', 'string']):
                    if field.id in df.columns:
                        group_field = field.id
                        break
            break

if not example_record_set_id or not numeric_field:
    print("No record set with a numeric field found.")
else:
    print(f"Using record set: {example_record_set_id}")
    print(f"Numeric field: {numeric_field}")
    if group_field:
        print(f"Grouping field: {group_field}")

    df = dataframes[example_record_set_id]
    # Remove missing values for numeric analysis
    filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce').notnull()]
    filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field])

    # Threshold filter example (use 10 or mean if low values)
    threshold = np.percentile(filtered_df[numeric_field], 75)
    filtered_high = filtered_df[filtered_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (75th percentile):")
    display(filtered_high.head())

    # Normalize
    filtered_high[f"{numeric_field}_normalized"] = (filtered_high[numeric_field] - filtered_high[numeric_field].mean()) / filtered_high[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_high[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    if group_field:
        group_means = filtered_high.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped {numeric_field} by {group_field} (mean):")
        display(group_means.head())

## 5. Visualization
Visualize distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and numeric_field:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna().astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {example_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10,5))
        # Show a boxplot by group
        gdf = df[[numeric_field, group_field]].dropna()
        gdf[numeric_field] = pd.to_numeric(gdf[numeric_field], errors='coerce')
        top_groups = gdf[group_field].value_counts().nlargest(5).index
        sns.boxplot(data=gdf[gdf[group_field].isin(top_groups)], x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field} (top 5 groups)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This exploration demonstrates the usage of the `mlcroissant` library to access, inspect, and analyze datasets annotated using the Croissant schema.

Through overview, EDA, and visualization using only entity `@id` references, users can identify patterns and meaningful statistics in scientific data packages. You are encouraged to explore additional fields, record sets, and relationships further, tailoring the analysis to your specific research questions.